In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import r2_score, mean_squared_error

# Reproducibility so synthetic data and fit are stable run-to-run
np.random.seed(42)


# ============================================================
# 1) PARAMETERS
# ============================================================
def get_params():
    # Media channels that get nonlinear carryover+saturation transforms
    media_channels = ["tv", "search", "social", "video"]

    # Non-media controls that enter linearly (after z-score scaling)
    non_media_channels = ["promo_index", "distribution_index", "seasonality_index"]

    # Adstock memory (higher = longer carryover)
    adstock_alpha = {"tv": 0.75, "search": 0.45, "social": 0.35, "video": 0.55}

    # Hill half-saturation point (higher = curve shifts right)
    hill_k = {"tv": 2.8, "search": 2.2, "social": 2.0, "video": 2.4}

    # Hill slope/shape (higher = stronger S-shape)
    hill_s = {"tv": 2.4, "search": 2.2, "social": 2.6, "video": 2.3}

    # Prior business beliefs for media effectiveness (Revenue / Spend) (ROAS <==> ROI)
    roi_priors_media = {"tv": 1.8, "search": 3.0, "social": 2.4, "video": 2.0}

    # Prior share split across non-media effects (must sum after normalization) ( % Contribution) ( 0.45 -> 45%)
    contribution_priors_non_media = {
        "promo_index": 0.45, 
        "distribution_index": 0.35,
        "seasonality_index": 0.20
    }

    # modelling 52w, 104w, 156w 
    return {
        "n_periods": 156,
        "freq": "W",
        "media_channels": media_channels,
        "non_media_channels": non_media_channels,
        "adstock_alpha": adstock_alpha,
        "hill_k": hill_k,
        "hill_s": hill_s,
        "roi_priors_media": roi_priors_media,
        "contribution_priors_non_media": contribution_priors_non_media,
        # Assumed total non-media contribution scale as fraction of average units
        "nonmedia_expected_share_of_units": 0.35,
        # Global prior strength in objective (higher = trust priors more)
        "l2_strength": 60.0,
        # Noise injected in synthetic unit generation
        "noise_sd_units": 1500
    }


# ============================================================
# 2) CORE TRANSFORMS
# ============================================================
def geometric_adstock(x, alpha):
    # Carryover recursion: current + alpha * previous stock
    out = np.zeros_like(x, dtype=float)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = x[t] + alpha * out[t - 1]
    return out


def hill_transform(x, k, s):
    # Saturation curve in [0,1], S-shaped when s>1
    x_safe = np.clip(x, 1e-9, None)
    num = np.power(x_safe, s)
    den = num + np.power(k, s)
    return num / den


def median_scale(x):
    # Robust scale for skewed media volumes
    med = np.median(x)
    return (x / med if med != 0 else x.copy()), med


def median_inverse(x_scaled, med):
    # Inverse of median scaling (traceability/debugging)
    return x_scaled * med


def zscore_scale(x):
    # Standardization: mean 0, std 1
    mu = np.mean(x)
    sd = np.std(x)
    sd = sd if sd > 0 else 1.0
    return (x - mu) / sd, mu, sd


def zscore_inverse(z, mu, sd):
    # Back-transform to original scale
    return z * sd + mu


# ============================================================
# 3) SYNTHETIC DATA GENERATION
# ============================================================
def generate_original_data(n_periods=156, freq="W", noise_sd_units=1500):
    idx = pd.date_range("2023-01-01", periods=n_periods, freq=freq)
    n = len(idx)
    t = np.arange(n)

    # Media impressions with seasonality + noise
    tv_imp = 15_000_000 + 2_000_000*np.sin(2*np.pi*t/52 + 0.3) + np.random.normal(0, 1_100_000, n)
    se_imp =  9_000_000 + 1_200_000*np.sin(2*np.pi*t/26 + 0.8) + np.random.normal(0,   700_000, n)
    so_imp =  7_000_000 + 1_100_000*np.sin(2*np.pi*t/13 + 1.2) + np.random.normal(0,   650_000, n)
    vi_imp =  8_000_000 + 1_300_000*np.sin(2*np.pi*t/39 + 0.5) + np.random.normal(0,   750_000, n)

    # Keep positive and realistic
    tv_imp = np.clip(tv_imp, 1_000_000, None)
    se_imp = np.clip(se_imp,   600_000, None)
    so_imp = np.clip(so_imp,   500_000, None)
    vi_imp = np.clip(vi_imp,   700_000, None)

    # Spend via CPM * impressions/1000
    tv_sp = (tv_imp / 1000.0) * np.random.normal(11.5, 0.7, n)
    se_sp = (se_imp / 1000.0) * np.random.normal(9.0, 0.5, n)
    so_sp = (so_imp / 1000.0) * np.random.normal(7.5, 0.4, n)
    vi_sp = (vi_imp / 1000.0) * np.random.normal(10.0, 0.6, n)

    # Non-media factors
    promo = np.clip(np.random.normal(0.5, 0.15, n), 0.05, 1.0)
    dist  = np.clip(0.75 + 0.05*np.sin(2*np.pi*t/52 + 2.2) + np.random.normal(0, 0.03, n), 0.55, 0.95)
    seas  = 0.5 + 0.5*np.sin(2*np.pi*t/52 - 1.1)

    # Unit price
    price = 9.5 + 0.35*np.sin(2*np.pi*t/52 + 0.2) + np.random.normal(0, 0.1, n)
    price = np.clip(price, 8.8, 10.5)

    # Ground-truth units (synthetic DGP)
    base_units = 120_000
    units = (
        base_units
        + 1800*np.log1p(tv_imp/1e6)
        + 2200*np.log1p(se_imp/1e6)
        + 1400*np.log1p(so_imp/1e6)
        + 1600*np.log1p(vi_imp/1e6)
        + 1500*((promo - promo.mean())/promo.std())
        + 1000*((dist  - dist.mean())/dist.std())
        + 1200*((seas  - seas.mean())/seas.std())
        + np.random.normal(0, noise_sd_units, n)
    )
    units = np.clip(units, 1000, None)
    revenue = units * price

    return pd.DataFrame({
        "date": idx,
        "tv_impressions": tv_imp, "tv_spend": tv_sp,
        "search_impressions": se_imp, "search_spend": se_sp,
        "social_impressions": so_imp, "social_spend": so_sp,
        "video_impressions": vi_imp, "video_spend": vi_sp,
        "promo_index": promo,
        "distribution_index": dist,
        "seasonality_index": seas,
        "price": price,
        "revenue": revenue
    })


# ============================================================
# 4) FEATURE ENGINEERING
# ============================================================
def build_features(df, p):
    out = df.copy()
    media_medians = {}
    non_media_stats = {}

    # target in units
    out["units"] = out["revenue"] / np.clip(out["price"], 1e-9, None)

    # media: median scaling -> adstock -> hill
    for c in p["media_channels"]:
        imp_col = f"{c}_impressions"
        scaled, med = median_scale(out[imp_col].values)
        media_medians[c] = med

        ads = geometric_adstock(scaled, p["adstock_alpha"][c])
        h = hill_transform(ads, p["hill_k"][c], p["hill_s"][c])

        out[f"{c}_scaled"] = scaled
        out[f"{c}_adstock"] = ads
        out[f"{c}_hill"] = h

    # non-media: zscore only
    for c in p["non_media_channels"]:
        z, mu, sd = zscore_scale(out[c].values)
        non_media_stats[c] = (mu, sd)
        out[f"{c}_scaled"] = z

    return out, media_medians, non_media_stats


# ============================================================
# 5) PRIORS (IN UNITS-SCALE COEFFICIENT SPACE)
# ============================================================
def build_prior_means_stds(df, p):
    media = p["media_channels"]
    non_media = p["non_media_channels"]
    avg_price = df["price"].mean()

    # Media priors from ROI beliefs
    mu_media, sd_media = [], []
    for c in media:
        roi = p["roi_priors_media"][c]               # revenue/spend
        units_per_spend = roi / avg_price            # convert to units/spend
        avg_spend = df[f"{c}_spend"].mean()
        avg_hill  = df[f"{c}_hill"].mean()
        mu = units_per_spend * (avg_spend / max(avg_hill, 1e-9)) # coeff prior
        sd = 0.5 # less will make closer to prior assumption
        mu_media.append(mu)
        sd_media.append(sd)

    # Non-media priors from expected contribution shares
    shares = p["contribution_priors_non_media"].copy()
    ssum = sum(shares.values())
    shares = {k: v / ssum for k, v in shares.items()}

    expected_nonmedia_units = p["nonmedia_expected_share_of_units"] * df["units"].mean()

    mu_non, sd_non = [], []
    for c in non_media:
        target_units = shares[c] * expected_nonmedia_units
        avg_abs_x = np.mean(np.abs(df[f"{c}_scaled"].values)) + 1e-9
        mu = target_units / avg_abs_x
        sd = 0.5
        mu_non.append(mu)
        sd_non.append(sd)

    return np.array(mu_media), np.array(sd_media), np.array(mu_non), np.array(sd_non)


# ============================================================
# 6) PRIOR-CENTERED RIDGE OBJECTIVE
# ============================================================
def ridge_prior_objective(beta, X, y, mu_prior, sigma_prior, l2_strength):
    # Data fit term (SSE)
    resid = y - X @ beta
    data_loss = np.sum(resid ** 2)

    # Prior term with per-coefficient precision = 1/sigma^2
    precision = 1.0 / np.clip(sigma_prior, 1e-9, None) ** 2
    prior_penalty = np.sum(precision * (beta - mu_prior) ** 2)

    # Total objective: data fit + weighted prior pull
    return data_loss + l2_strength * prior_penalty


# ============================================================
# 7) FIT: TARGET Z-SCORE MODELING + BACK-CONVERSION
# ============================================================
def fit_prior_centered_ridge(df, p):
    media = p["media_channels"]
    non_media = p["non_media_channels"]

    # --- Target transform: model on z-scored sales(units) --- (non revenue modelling  where ur target kpi is not in terms of $)
    y_units = df["units"].values
    y_mu = y_units.mean()
    y_sd = y_units.std(ddof=0)
    if y_sd < 1e-9:
        y_sd = 1.0
    y_z = (y_units - y_mu) / y_sd

    # Design matrix (intercept + media transformed + non-media z-scored)
    X_media = np.column_stack([df[f"{c}_hill"].values for c in media])
    X_non = np.column_stack([df[f"{c}_scaled"].values for c in non_media])
    X_core = np.column_stack([X_media, X_non])
    X = np.column_stack([np.ones(len(df)), X_core])

    # Priors first in UNITS coefficient scale
    mu_m_u, sd_m_u, mu_n_u, sd_n_u = build_prior_means_stds(df, p)
    mu_intercept_u = np.array([y_units.mean()])
    sd_intercept_u = np.array([10.0 * y_units.std(ddof=0) + 1e-6])

    mu_prior_u = np.concatenate([mu_intercept_u, mu_m_u, mu_n_u])
    sd_prior_u = np.concatenate([sd_intercept_u, sd_m_u, sd_n_u])

    # Convert priors from units-coef scale -> z-target coef scale
    # y_z = (y_units - y_mu)/y_sd
    # b0_z = (b0_u - y_mu)/y_sd ; bj_z = bj_u / y_sd
    mu_prior_z = mu_prior_u.copy()
    mu_prior_z[0] = (mu_prior_u[0] - y_mu) / y_sd
    mu_prior_z[1:] = mu_prior_u[1:] / y_sd

    sd_prior_z = np.clip(sd_prior_u / y_sd, 1e-9, None)

    # Start optimization from prior means in z-space
    x0 = mu_prior_z.copy()

    # Optional positivity for media coefficients in z-space (equivalent sign in units)
    n_media = len(media)
    n_non = len(non_media)
    bounds = [(None, None)] + [(0.0, None)] * n_media + [(None, None)] * n_non

    # Optimize objective in z-space
    res = minimize(
        ridge_prior_objective,
        x0=x0,
        args=(X, y_z, mu_prior_z, sd_prior_z, p["l2_strength"]),
        method="L-BFGS-B",
        bounds=bounds
    )
    if not res.success:
        raise RuntimeError(f"Optimization failed: {res.message}")

    beta_z = res.x
    y_hat_z = X @ beta_z

    # Convert coefficients back to units-space for business interpretation
    # b0_u = y_mu + y_sd*b0_z ; bj_u = y_sd*bj_z
    beta_u = beta_z.copy()
    beta_u[0] = y_mu + y_sd * beta_z[0]
    beta_u[1:] = y_sd * beta_z[1:]

    # Units predictions from back-converted coefficients
    y_hat_units = X @ beta_u

    # Metrics in both scales
    metrics = {
        "r2_z": r2_score(y_z, y_hat_z),
        "rmse_z": np.sqrt(mean_squared_error(y_z, y_hat_z)),
        "r2_units": r2_score(y_units, y_hat_units),
        "rmse_units": np.sqrt(mean_squared_error(y_units, y_hat_units))
    }

    intercept = beta_u[0]
    b_media = beta_u[1:1 + len(media)]
    b_non = beta_u[1 + len(media):]

    return (
        intercept, b_media, b_non, y_hat_units, metrics,
        mu_prior_u, sd_prior_u,
        {"y_mu": y_mu, "y_sd": y_sd, "beta_z": beta_z}
    )


# ============================================================
# 8) CONTRIBUTIONS + OUTPUT TABLES
# ============================================================
def build_outputs(df, p, intercept, b_media, b_non):
    out = df.copy()
    media = p["media_channels"]
    non_media = p["non_media_channels"]

    # contributions in units-space directly
    for i, c in enumerate(media):
        out[f"contrib_{c}_units"] = b_media[i] * out[f"{c}_hill"].values

    for i, c in enumerate(non_media):
        out[f"contrib_{c}_units"] = b_non[i] * out[f"{c}_scaled"].values

    out["baseline_units"] = intercept
    contrib_unit_cols = [f"contrib_{c}_units" for c in media + non_media]
    out["incremental_units"] = out[contrib_unit_cols].sum(axis=1)
    out["predicted_units"] = out["baseline_units"] + out["incremental_units"]

    for c in media + non_media:
        out[f"contrib_{c}_revenue"] = out[f"contrib_{c}_units"] * out["price"]

    out["baseline_revenue"] = out["baseline_units"] * out["price"]
    out["incremental_revenue"] = out[[f"contrib_{c}_revenue" for c in media + non_media]].sum(axis=1)
    out["predicted_revenue"] = out["baseline_revenue"] + out["incremental_revenue"]

    weekly_cols = ["date", "price", "revenue", "predicted_revenue", "baseline_revenue", "incremental_revenue"]
    for c in media + non_media:
        weekly_cols += [f"contrib_{c}_units", f"contrib_{c}_revenue"]
    weekly_df = out[weekly_cols].copy()

    roi_rows = []
    for c in media:
        inc_rev = out[f"contrib_{c}_revenue"].sum()
        spend = out[f"{c}_spend"].sum()
        roi_rows.append({
            "channel": c, "type": "media",
            "total_spend": spend,
            "total_incremental_revenue": inc_rev,
            "roi": (inc_rev / spend) if spend > 0 else np.nan
        })

    for c in non_media:
        roi_rows.append({
            "channel": c, "type": "non_media",
            "total_spend": np.nan,
            "total_incremental_revenue": out[f"contrib_{c}_revenue"].sum(),
            "roi": np.nan
        })

    roi_df = pd.DataFrame(roi_rows).sort_values(["type", "channel"]).reset_index(drop=True)
    return out, weekly_df, roi_df


# ============================================================
# 9) RESPONSE CURVES
# ============================================================
def response_curve_media_channel(df, p, channel, beta_media, n_points=60):
    imp_col = f"{channel}_impressions"
    spend_col = f"{channel}_spend"

    # Explore spend/impression range around historical support
    p5, p95 = np.percentile(df[imp_col].values, [5, 95])
    imp_grid = np.linspace(max(1.0, 0.5 * p5), 1.5 * p95, n_points)

    # Apply same media transforms as train (steady-state adstock approximation)
    med = np.median(df[imp_col].values)
    x_sc = imp_grid / med if med != 0 else imp_grid
    x_ads = x_sc / (1 - p["adstock_alpha"][channel])
    x_hill = hill_transform(x_ads, p["hill_k"][channel], p["hill_s"][channel])

    # Units then revenue using avg price
    incr_units = beta_media * x_hill
    incr_revenue = incr_units * df["price"].mean()

    # Spend from avg CPM
    avg_cpm = np.mean((df[spend_col].values / np.clip(df[imp_col].values, 1e-9, None)) * 1000.0)
    spend_grid = (imp_grid / 1000.0) * avg_cpm

    roi = incr_revenue / np.clip(spend_grid, 1e-9, None)

    # mROI = local derivative dRevenue/dSpend (finite diff)
    d_rev = np.diff(incr_revenue)
    d_spend = np.diff(spend_grid)
    mroi_mid = d_rev / np.clip(d_spend, 1e-9, None)

    # Pad to full length
    mroi = np.empty_like(spend_grid)
    mroi[0] = mroi_mid[0]
    mroi[-1] = mroi_mid[-1]
    if len(mroi) > 2:
        mroi[1:-1] = 0.5 * (mroi_mid[:-1] + mroi_mid[1:])

    return pd.DataFrame({
        "impressions": imp_grid,
        "spend": spend_grid,
        "predicted_incremental_units": incr_units,
        "predicted_incremental_revenue": incr_revenue,
        "implied_roi": roi,
        "mroi": mroi
    })


def write_response_curves_excel(df, p, b_media, out_file="./outputs/response_curves.xlsx"):
    with pd.ExcelWriter(out_file, engine="xlsxwriter") as writer:
        for i, c in enumerate(p["media_channels"]):
            curve = response_curve_media_channel(df, p, c, b_media[i], n_points=60)
            curve.to_excel(writer, sheet_name=c[:31], index=False)


# ============================================================
# 10) MAIN
# ============================================================
def run_pipeline():
    p = get_params()

    # Synthetic source data
    original = generate_original_data(
        n_periods=p["n_periods"],
        freq=p["freq"],
        noise_sd_units=p["noise_sd_units"]
    )
    original.to_csv("./outputs/original_data.csv", index=False)

    # Build transformed features
    feat, media_medians, non_media_stats = build_features(original, p)

    # Fit model with z-scored target, return unit-scale coefficients
    intercept, b_media, b_non, y_hat, metrics, mu_prior, sd_prior, model_info = fit_prior_centered_ridge(feat, p)

    # Attribution outputs
    full_df, weekly_df, roi_df = build_outputs(feat, p, intercept, b_media, b_non)
    weekly_df.to_csv("./outputs/weekly_contribution.csv", index=False)
    roi_df.to_csv("./outputs/roi.csv", index=False)
    write_response_curves_excel(full_df, p, b_media, out_file="./outputs/response_curves.xlsx")

    print("Saved files:")
    print("- original_data.csv")
    print("- weekly_contribution.csv")
    print("- roi.csv")
    print("- response_curves.xlsx")


if __name__ == "__main__":
    run_pipeline()

Saved files:
- original_data.csv
- weekly_contribution.csv
- roi.csv
- response_curves.xlsx


In [4]:
# Run seperate modules

In [5]:
p = get_params()

In [6]:
p

{'n_periods': 156,
 'freq': 'W',
 'media_channels': ['tv', 'search', 'social', 'video'],
 'non_media_channels': ['promo_index',
  'distribution_index',
  'seasonality_index'],
 'adstock_alpha': {'tv': 0.75, 'search': 0.45, 'social': 0.35, 'video': 0.55},
 'hill_k': {'tv': 2.8, 'search': 2.2, 'social': 2.0, 'video': 2.4},
 'hill_s': {'tv': 2.4, 'search': 2.2, 'social': 2.6, 'video': 2.3},
 'roi_priors_media': {'tv': 1.8, 'search': 3.0, 'social': 2.4, 'video': 2.0},
 'contribution_priors_non_media': {'promo_index': 0.45,
  'distribution_index': 0.35,
  'seasonality_index': 0.2},
 'nonmedia_expected_share_of_units': 0.35,
 'l2_strength': 60.0,
 'noise_sd_units': 1500}

In [7]:
original = generate_original_data(
    n_periods=p["n_periods"],
    freq=p["freq"],
    noise_sd_units=p["noise_sd_units"]
)

In [8]:
original

,date,tv_impressions,tv_spend,search_impressions,search_spend,social_impressions,social_spend,video_impressions,video_spend,promo_index,distribution_index,seasonality_index,price,revenue
0,2023-01-01,1.525220e+07,172421.971647,1.000759e+07,92624.198716,6.587533e+06,51818.561010,9.147375e+06,98186.544480,0.427466,0.825531,0.054396,9.621759,1.328036e+06
1,2023-01-08,1.667466e+07,188493.199273,9.691747e+06,92219.788973,9.089775e+06,66175.234559,9.113918e+06,89723.676570,0.408175,0.790612,0.084983,9.600583,1.298431e+06
2,2023-01-15,1.747246e+07,210600.102270,1.071574e+07,92917.080533,6.984868e+06,52060.274411,9.321466e+06,95186.172853,0.573681,0.784596,0.121621,9.876848,1.373402e+06
3,2023-01-22,1.776542e+07,208531.120876,9.517259e+06,89721.695801,7.345943e+06,53227.061709,8.687520e+06,83902.236727,0.446296,0.793985,0.163777,9.775557,1.341147e+06
4,2023-01-29,1.579289e+07,187926.368541,1.051051e+07,88856.200120,6.730126e+06,55157.023553,7.568609e+06,67418.431838,0.479090,0.746269,0.210836,9.778355,1.358557e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,2025-11-23,1.296133e+07,136302.664791,8.528295e+06,72473.217713,6.342161e+06,44759.694108,7.091658e+06,71986.596268,0.369056,0.841520,0.004439,9.240077,1.226440e+06
152,2025-11-30,1.131394e+07,130234.037000,8.667924e+06,77597.579958,6.589185e+06,47245.737807,7.443052e+06,73562.946137,0.632211,0.812795,0.000039,9.409284,1.275681e+06
153,2025-12-07,1.507733e+07,182912.081795,8.839724e+06,71706.754689,6.760245e+06,49056.150879,9.104404e+06,86915.425762,0.608170,0.831272,0.002930,9.314027,1.289277e+06
154,2025-12-14,1.709717e+07,185753.457199,9.247420e+06,82854.106982,7.258959e+06,52906.264497,7.287525e+06,77460.739769,0.362559,0.810288,0.013068,9.415853,1.274777e+06


In [9]:
feat, media_medians, non_media_stats = build_features(original, p)

In [27]:
def fit_prior_centered_ridge(df, p):
    media = p["media_channels"]
    non_media = p["non_media_channels"]

    # --- Target transform: model on z-scored sales(units) --- (non revenue modelling  where ur target kpi is not in terms of $)
    y_units = df["units"].values
    y_mu = y_units.mean()
    y_sd = y_units.std(ddof=0)
    if y_sd < 1e-9:
        y_sd = 1.0
    y_z = (y_units - y_mu) / y_sd

    # Design matrix (intercept + media transformed + non-media z-scored)
    X_media = np.column_stack([df[f"{c}_hill"].values for c in media])

    X_non = np.column_stack([df[f"{c}_scaled"].values for c in non_media])

   

    X_core = np.column_stack([X_media, X_non])
    X = np.column_stack([np.ones(len(df)), X_core])

    

    # Priors first in UNITS coefficient scale
    mu_m_u, sd_m_u, mu_n_u, sd_n_u = build_prior_means_stds(df, p)
    mu_intercept_u = np.array([y_units.mean()])
    sd_intercept_u = np.array([10.0 * y_units.std(ddof=0) + 1e-6])

    mu_prior_u = np.concatenate([mu_intercept_u, mu_m_u, mu_n_u])
    sd_prior_u = np.concatenate([sd_intercept_u, sd_m_u, sd_n_u])

    # Convert priors from units-coef scale -> z-target coef scale
    # y_z = (y_units - y_mu)/y_sd
    # b0_z = (b0_u - y_mu)/y_sd ; bj_z = bj_u / y_sd
    mu_prior_z = mu_prior_u.copy()
    mu_prior_z[0] = (mu_prior_u[0] - y_mu) / y_sd
    mu_prior_z[1:] = mu_prior_u[1:] / y_sd

    sd_prior_z = np.clip(sd_prior_u / y_sd, 1e-9, None)

    return sd_prior_z

    # Start optimization from prior means in z-space
    x0 = mu_prior_z.copy()

    # Optional positivity for media coefficients in z-space (equivalent sign in units)
    n_media = len(media)
    n_non = len(non_media)
    bounds = [(None, None)] + [(0.0, None)] * n_media + [(None, None)] * n_non

    # Optimize objective in z-space
    res = minimize(
        ridge_prior_objective,
        x0=x0,
        args=(X, y_z, mu_prior_z, sd_prior_z, p["l2_strength"]),
        method="L-BFGS-B",
        bounds=bounds
    )
    if not res.success:
        raise RuntimeError(f"Optimization failed: {res.message}")

    beta_z = res.x
    y_hat_z = X @ beta_z

    # Convert coefficients back to units-space for business interpretation
    # b0_u = y_mu + y_sd*b0_z ; bj_u = y_sd*bj_z
    beta_u = beta_z.copy()
    beta_u[0] = y_mu + y_sd * beta_z[0]
    beta_u[1:] = y_sd * beta_z[1:]

    # Units predictions from back-converted coefficients
    y_hat_units = X @ beta_u

    # Metrics in both scales
    metrics = {
        "r2_z": r2_score(y_z, y_hat_z),
        "rmse_z": np.sqrt(mean_squared_error(y_z, y_hat_z)),
        "r2_units": r2_score(y_units, y_hat_units),
        "rmse_units": np.sqrt(mean_squared_error(y_units, y_hat_units))
    }

    intercept = beta_u[0]
    b_media = beta_u[1:1 + len(media)]
    b_non = beta_u[1 + len(media):]

    return (
        intercept, b_media, b_non, y_hat_units, metrics,
        mu_prior_u, sd_prior_u,
        {"y_mu": y_mu, "y_sd": y_sd, "beta_z": beta_z}
    )

In [28]:
fit_prior_centered_ridge(feat, p)

array([1.00000000e+01, 2.16598938e-04, 2.16598938e-04, 2.16598938e-04,
       2.16598938e-04, 2.16598938e-04, 2.16598938e-04, 2.16598938e-04])